In [0]:
%python
# --- DATABRICKS KAPPA-STREAMING SILVER LAYER ---
from pyspark.sql.functions import col, lit, sha2, levenshtein, when, hour

# 1. Define paths using Unity Catalog
# In Community Edition, 'hive_metastore' is your only Catalog
catalog = "hive_metastore"
bronze_schema = "prism_bronze"
silver_schema = "prism_silver"

# Corrected table paths
source_table = f"{catalog}.{bronze_schema}.transactions_raw"
target_table = f"{catalog}.{silver_schema}.transactions_refined"

print(f"Reading from: {source_table}")
# This will list all tables in your bronze schema
try:
    df_check = spark.table(source_table)
    print(f"✅ Success! Found {df_check.count()} rows in Bronze.")
except Exception as e:
    print(f"❌ Table not found. Run your Bronze Ingestion notebook first and ensure it saves to {source_table}")

# 2. Start the Streaming 'Evasion Hunter'
query = (spark.readStream
    .table(source_table)
    .withColumn("hashed_user_id", sha2(col("user_id").cast("string"), 256))
    .drop("user_id")
    .withColumn("edit_distance", levenshtein(col("counterparty"), lit("NORTH_STAR_SHIPPING")))
    .withColumn("preliminary_risk_score", 
        when(col("edit_distance") < 8, 0.95)
        .when(col("country") == "IRAN_PROXY", 0.70)
        .otherwise(0.01))
    .writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .toTable(target_table)
)